# Strands 에이전트와 AgentCore 메모리 (단기 메모리)


## 소개

이 튜토리얼은 Strands 에이전트와 AgentCore **단기 메모리** (원시 이벤트)를 사용하여 **개인 에이전트**를 구축하는 방법을 보여줍니다. 에이전트는 `get_last_k_turns`를 사용하여 세션 내 최근 대화를 기억하고, 사용자가 돌아왔을 때 대화를 원활하게 이어갈 수 있습니다.


### 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 유형       | 개인 에이전트                                                                    |
| 에이전트 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, AgentInitializedEvent 및 MessageAddedEvent 훅             |
| 예제 난이도         | 초급                                                                             |

학습 내용:
- 대화 연속성을 위한 단기 메모리 사용
- 최근 K개의 대화 턴 조회
- 실시간 정보를 위한 웹 검색 도구
- 대화 기록으로 에이전트 초기화

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항

- Python 3.10+
- AgentCore 메모리 권한이 있는 AWS 자격 증명
- AgentCore 메모리 역할 ARN
- Amazon Bedrock 모델 접근 권한

환경 설정부터 시작하겠습니다!

## 1단계: 설정 및 임포트

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime

# 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("personal-agent")

In [ ]:
# 임포트
import os
from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory import MemoryClient

# 설정
REGION = os.getenv('AWS_REGION', 'us-west-2') # 에이전트의 AWS 리전
ACTOR_ID = "user_123" # 고유 식별자 (에이전트 ID, 사용자 ID 등)
SESSION_ID = "personal_session_001" # 고유 세션 식별자


## 2단계: 웹 검색 도구

먼저 에이전트를 위한 간단한 웹 검색 도구를 만들어 보겠습니다.

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "결과를 찾을 수 없습니다."
    except RatelimitException:
        return "요청 제한에 도달했습니다. 나중에 다시 시도해 주세요."
    except DDGSException as e:
        return f"검색 오류: {e}"
    except Exception as e:
        return f"검색 오류: {str(e)}"

logger.info("웹 검색 도구 준비 완료")

## 3단계: 메모리 리소스 생성
단기 메모리의 경우, 전략 없이 메모리 리소스를 생성합니다. 이렇게 하면 `get_last_k_turns`로 조회할 수 있는 원시 대화 턴이 저장됩니다.


In [ ]:
from botocore.exceptions import ClientError

# 메모리 클라이언트 초기화
client = MemoryClient(region_name=REGION)
memory_name = "PersonalAgentMemory"

try:
    # 전략 없이 메모리 리소스 생성 (단기 메모리만 접근 가능)
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리를 위해 전략 없음
        description="Short-term memory for personal agent",
        event_expiry_days=7, # 단기 메모리 보존 기간. 최대 365일까지 설정 가능.
    )
    memory_id = memory['id']
    logger.info(f"메모리 생성 완료: {memory_id}")
except ClientError as e:
    logger.info(f"오류: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하는 경우, ID를 조회
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 표시
    logger.error(f"오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")

## 4단계: 메모리 훅

이 단계에서는 메모리 작업을 자동화하는 커스텀 `MemoryHookProvider` 클래스를 정의합니다. 훅은 에이전트 실행 생명주기의 특정 시점에 실행되는 특수 함수입니다. 여기서 만드는 메모리 훅은 두 가지 주요 기능을 수행합니다:
1. **최근 대화 로드**: `AgentInitializedEvent` 훅이 에이전트 초기화 시 자동으로 최근 대화 기록을 로드합니다.
2. **마지막 메시지 저장**: 새로운 대화 메시지를 저장합니다.

이를 통해 수동 관리 없이 원활한 메모리 경험을 제공합니다.

In [ ]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_client: MemoryClient, memory_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 기록을 로드"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            
            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return
            
            # 메모리에서 최근 5개의 대화 턴 로드
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=self.memory_id,
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )
            
            if recent_turns:
                # 대화 기록을 컨텍스트로 포맷팅
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role']
                        content = message['content']['text']
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                # 에이전트의 시스템 프롬프트에 컨텍스트 추가
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info(f"대화 턴 {len(recent_turns)}개 로드 완료")
                
        except Exception as e:
            logger.error(f"메모리 로드 오류: {e}")
    
    def on_message_added(self, event: MessageAddedEvent):
        """메모리에 메시지 저장"""
        messages = event.agent.messages
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if messages[-1]["content"][0].get("text"):
                self.memory_client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=[(messages[-1]["content"][0]["text"], messages[-1]["role"])]
                )
        except Exception as e:
            logger.error(f"메모리 저장 오류: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        # 메모리 훅 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 5단계: 웹 검색 기능을 갖춘 개인 에이전트 생성

In [ ]:
def create_personal_agent():
    """메모리와 웹 검색 기능을 갖춘 개인 에이전트 생성"""
    # 시스템 프롬프트: 웹 검색 기능을 갖춘 친절하고 전문적인 개인 비서
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        system_prompt=f"""You are a helpful personal assistant with web search capabilities.
        
        You can help with:
        - General questions and information lookup
        - Web searches for current information
        - Personal task management
        
        When you need current information, use the websearch function.
        Today's date: {datetime.today().strftime('%Y-%m-%d')}
        Be friendly and professional.""",
        hooks=[MemoryHookProvider(client, memory_id)],
        tools=[websearch],
        state={"actor_id": ACTOR_ID, "session_id": SESSION_ID}
    )
    return agent

# 에이전트 생성
agent = create_personal_agent()
logger.info("메모리와 웹 검색 기능을 갖춘 개인 에이전트 생성 완료")

#### 축하합니다! 에이전트가 준비되었습니다! :) 
## 에이전트를 테스트해 봅시다

In [ ]:
# 메모리를 사용한 대화 테스트
print("=== 첫 번째 대화 ===")
# 사용자: 제 이름은 Alex이고 AI에 대해 배우고 싶습니다.
print(f"사용자: My name is Alex and I'm interested in learning about AI.")
print(f"에이전트: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

In [ ]:
# 사용자: 2025년 최신 AI 트렌드를 검색해 줄 수 있나요?
print(f"사용자: Can you search for the latest AI trends in 2025?")
print(f"에이전트: ", end="")
agent("Can you search for the latest AI trends in 2025?")

In [ ]:
# 사용자: 특히 머신러닝 응용 분야에 관심이 있습니다.
print(f"사용자: I'm particularly interested in machine learning applications.")
print(f"에이전트: ", end="")
agent("I'm particularly interested in machine learning applications.")

## 메모리 연속성 테스트

메모리 시스템이 올바르게 작동하는지 테스트하기 위해, 새로운 에이전트 인스턴스를 생성하고 이전에 저장된 정보에 접근할 수 있는지 확인해 보겠습니다:

In [ ]:
# 새 에이전트 인스턴스 생성 (사용자가 돌아온 상황 시뮬레이션)
print("=== 사용자 복귀 - 새 세션 ===")
new_agent = create_personal_agent()

# 메모리 연속성 테스트
# 사용자: 제 이름이 뭐였죠?
print(f"사용자: What was my name again?")
print(f"에이전트: ", end="")
new_agent("What was my name again?")

# 사용자: 머신러닝에 대해 더 많은 정보를 검색해 줄 수 있나요?
print(f"사용자: Can you search for more information about machine learning?")
print(f"에이전트: ", end="")
new_agent("Can you search for more information about machine learning?")

## 저장된 메모리 확인

In [ ]:
# 메모리에 저장된 내용 확인
print("=== 메모리 내용 ===")
recent_turns = client.get_last_k_turns(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
    k=3 # k 값을 조정하여 더 많거나 적은 턴을 확인
)

for i, turn in enumerate(recent_turns, 1):
    print(f"턴 {i}:")
    for message in turn:
        role = message['role']
        content = message['content']['text'][:100] + "..." if len(message['content']['text']) > 100 else message['content']['text']
        print(f"  {role}: {content}")
    print()

## 요약

이 튜토리얼에서는 개인 에이전트를 구축하는 방법을 보여주었습니다. 학습한 내용:

- 전략 없이 메모리 리소스 생성
- `get_last_k_turns`를 사용한 대화 기록 조회
- 에이전트에 웹 검색 기능 추가
- 컨텍스트 로딩을 위한 메모리 훅 구현

**다음 단계:**
- 더 정교한 도구 추가
- 장기 메모리 전략 구현
- 다중 소스를 활용한 검색 기능 강화

## 정리 (선택사항)

In [ ]:
# 메모리 리소스를 삭제하려면 주석을 해제하세요
# client.delete_memory_and_wait(memory_id)
# logger.info(f"메모리 삭제 완료: {memory_id}")